# Retail Intelligence Business Analytics

## Notebook 3: Data Cleaning & Feature Engineering

### Objective

The objective of this notebook is to clean the Global Superstore dataset, handle data quality issues, validate business rules, engineer new analytical features, and prepare a high-quality dataset for SQL analysis and Power BI dashboard development.

In [2]:
import pandas as pd
import numpy as np

# Load Orders sheet
df = pd.read_excel(
    "../data/raw/Global Superstore Data.xlsx",
    sheet_name="Orders"
)

## 1. Missing Value Treatment

This step addresses missing values identified during the data quality assessment. Appropriate handling ensures the dataset remains reliable without introducing bias into future analyses.

In [3]:
df.isnull().sum()

Row ID                0
Order ID              0
Order Date            0
Ship Date             0
Ship Mode             0
Customer ID           0
Customer Name         0
Segment               0
Postal Code       41296
City                  0
State                 0
Country               0
Region                0
Market                0
Product ID            0
Product Name          0
Sub-Category          0
Category              0
Sales                 0
Quantity              0
Discount              0
Profit                0
Shipping Cost         0
Order Priority        0
dtype: int64

In [4]:
df["Postal Code"] = df["Postal Code"].fillna("Not Available")

In [5]:
df["Postal Code"].isna().sum()

np.int64(0)

### Observations

- Postal Code no longer contains missing values.
- No records were removed.
- Business information remains unchanged.

## 2. Date Validation

Business rules require that every order is shipped after it is placed. This step validates chronological consistency and calculates delivery duration for each transaction.

In [6]:
invalid_dates = df[df["Ship Date"] < df["Order Date"]]

invalid_dates

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Postal Code,City,...,Product ID,Product Name,Sub-Category,Category,Sales,Quantity,Discount,Profit,Shipping Cost,Order Priority


In [7]:
print(f"Invalid Records : {len(invalid_dates)}")

Invalid Records : 0


### Observations

- No invalid shipping dates were detected.
- Date columns are consistent and ready for analysis.

## 3. Feature Engineering

Additional business features are created to simplify analytical reporting, improve dashboard development, and support deeper business insights.

In [8]:
# Delivery time
df["Delivery Days"] = (
    df["Ship Date"] - df["Order Date"]
).dt.days

In [9]:
# Profit percentage
df["Profit Margin (%)"] = (
    (df["Profit"] / df["Sales"]) * 100
).round(2)

In [10]:
# Loss-making orders
df["Loss Order"] = np.where(
    df["Profit"] < 0,
    "Yes",
    "No"
)

In [11]:
# Discount categories
df["Discount Band"] = pd.cut(
    df["Discount"],
    bins=[-0.01, 0, 0.20, 0.50, 1],
    labels=[
        "No Discount",
        "Low",
        "Medium",
        "High"
    ]
)

In [12]:
# Sales categories
df["Sales Category"] = pd.cut(
    df["Sales"],
    bins=[0,100,500,1000,float("inf")],
    labels=[
        "Low",
        "Medium",
        "High",
        "Premium"
    ]
)

In [13]:
# Order year
df["Order Year"] = df["Order Date"].dt.year

In [14]:
# Preview newly created features
df[
    [
        "Order Date",
        "Ship Date",
        "Sales",
        "Profit",
        "Delivery Days",
        "Profit Margin (%)",
        "Loss Order",
        "Discount Band",
        "Sales Category",
        "Order Year"
    ]
].head()

,Order Date,Ship Date,Sales,Profit,Delivery Days,Profit Margin (%),Loss Order,Discount Band,Sales Category,Order Year
0,2017-03-22,2017-03-29,731.82,102.42,7,14.00,No,No Discount,High,2017
1,2015-09-01,2015-09-04,243.54,104.49,3,42.90,No,No Discount,Medium,2015
2,2017-03-22,2017-03-29,346.32,13.77,7,3.98,No,No Discount,Medium,2017
3,2017-03-22,2017-03-29,169.68,79.68,7,46.96,No,No Discount,Medium,2017
4,2015-09-01,2015-09-04,203.88,24.36,3,11.95,No,No Discount,Medium,2015


### Observations

- Delivery duration is now available for each order.
- Profit Margin provides a better measure of profitability than profit alone.
- Loss-making orders can now be identified quickly.
- Discount and sales values have been grouped into meaningful business categories.
- Order Year will simplify yearly trend analysis.

## 4. Final Validation

Perform one final validation to confirm that all cleaning operations and engineered features have been applied successfully before exporting the processed dataset.

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 30 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Row ID             51290 non-null  int64         
 1   Order ID           51290 non-null  object        
 2   Order Date         51290 non-null  datetime64[ns]
 3   Ship Date          51290 non-null  datetime64[ns]
 4   Ship Mode          51290 non-null  object        
 5   Customer ID        51290 non-null  object        
 6   Customer Name      51290 non-null  object        
 7   Segment            51290 non-null  object        
 8   Postal Code        51290 non-null  object        
 9   City               51290 non-null  object        
 10  State              51290 non-null  object        
 11  Country            51290 non-null  object        
 12  Region             51290 non-null  object        
 13  Market             51290 non-null  object        
 14  Produc

In [16]:
df.sample(5, random_state=42)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Postal Code,City,...,Discount,Profit,Shipping Cost,Order Priority,Delivery Days,Profit Margin (%),Loss Order,Discount Band,Sales Category,Order Year
49728,39143,CA-2015-JK15205140-42274,2015-09-27,2015-10-02,Standard Class,JK-152051408,Jamie Kunitz,Consumer,22204.0,Arlington,...,0.0,561.5640,117.95,Medium,5,30.00,No,No Discount,Premium,2015
45547,36842,CA-2017-MG17890140-43001,2017-09-23,2017-09-27,Standard Class,MG-178901406,Michael Granlund,Home Office,3301.0,Concord,...,0.0,17.5240,6.11,Medium,4,26.00,No,No Discount,Low,2017
15664,11413,ES-2017-PT1909048-42954,2017-08-07,2017-08-10,First Class,PT-1909048,Pete Takahito,Consumer,Not Available,Aschaffenburg,...,0.0,72.0000,108.11,Medium,3,10.99,No,No Discount,High,2017
40561,35001,CA-2015-DH13075140-42168,2015-06-13,2015-06-19,Standard Class,DH-130751408,Dave Hallsten,Corporate,35601.0,Decatur,...,0.0,314.9895,73.62,Medium,6,35.00,No,No Discount,High,2015
49426,39371,CA-2017-JM15250140-42737,2017-01-02,2017-01-06,Standard Class,JM-152501402,Janet Martin,Consumer,77340.0,Huntsville,...,0.8,-22.6842,1.90,Medium,4,-165.00,Yes,High,Low,2017


In [17]:
df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Row ID,51290.0,NaN,NaN,NaN,25645.5,1.0,12823.25,25645.5,38467.75,51290.0,14806.29199
Order ID,51290,25728,CA-2017-SV20365140-42999,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order Date,51290,NaN,NaN,NaN,2016-05-11 11:52:18.436342272,2014-01-01 00:00:00,2015-06-19 00:00:00,2016-07-08 00:00:00,2017-05-22 00:00:00,2017-12-31 00:00:00,NaN
Ship Date,51290,NaN,NaN,NaN,2016-05-15 11:09:41.306297344,2014-01-03 00:00:00,2015-06-23 00:00:00,2016-07-12 00:00:00,2017-05-26 00:00:00,2018-01-07 00:00:00,NaN
Ship Mode,51290,4,Standard Class,30775,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer ID,51290,17415,SV-203651406,26,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer Name,51290,796,Muhammed Yedwab,108,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Segment,51290,3,Consumer,26518,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Postal Code,51290,632,Not Available,41296,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,51290,3650,New York City,915,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Observations

- Newly created features have been added successfully.
- The dataset is complete and ready for analysis.
- No further cleaning is required at this stage.

## 5. Export Processed Dataset

Export the cleaned and feature-engineered dataset for use in SQL analysis, Power BI dashboard development, and future analytical tasks.

In [18]:
df.to_csv(
    "../data/processed/cleaned_orders.csv",
    index=False
)

print("Processed dataset exported successfully.")

Processed dataset exported successfully.


In [19]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Postal Code,City,...,Discount,Profit,Shipping Cost,Order Priority,Delivery Days,Profit Margin (%),Loss Order,Discount Band,Sales Category,Order Year
0,24599,IN-2017-CA120551-42816,2017-03-22,2017-03-29,Standard Class,CA-120551,Cathy Armstrong,Home Office,Not Available,Herat,...,0.0,102.42,39.66,Medium,7,14.00,No,No Discount,High,2017
1,29465,ID-2015-BD116051-42248,2015-09-01,2015-09-04,Second Class,BD-116051,Brian Dahlen,Consumer,Not Available,Herat,...,0.0,104.49,18.72,Medium,3,42.90,No,No Discount,Medium,2015
2,24598,IN-2017-CA120551-42816,2017-03-22,2017-03-29,Standard Class,CA-120551,Cathy Armstrong,Home Office,Not Available,Herat,...,0.0,13.77,14.10,Medium,7,3.98,No,No Discount,Medium,2017
3,24597,IN-2017-CA120551-42816,2017-03-22,2017-03-29,Standard Class,CA-120551,Cathy Armstrong,Home Office,Not Available,Herat,...,0.0,79.68,11.01,Medium,7,46.96,No,No Discount,Medium,2017
4,29464,ID-2015-BD116051-42248,2015-09-01,2015-09-04,Second Class,BD-116051,Brian Dahlen,Consumer,Not Available,Herat,...,0.0,24.36,5.72,Medium,3,11.95,No,No Discount,Medium,2015


# Summary

### Tasks Completed

- Missing values were successfully handled.
- Date consistency was validated.
- Business rules were verified.
- New analytical features were engineered.
- Final validation confirmed data quality.
- The cleaned dataset was exported successfully.

The processed dataset is now ready for exploratory data analysis, SQL-based business intelligence, and interactive Power BI dashboard development.